# 03 – Monte Carlo VaR

**Goal:** Run Monte Carlo VaR using GBM for path counts of 1 000, 5 000,
and 10 000.  Compare VaR estimates and latency against the parametric baseline.

**Prerequisite:** Notebooks 01 and 02 must have been run.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

import config
from src.data_loader      import load_saved_data
from src.preprocess       import compute_log_returns
from src.portfolio        import initialize_portfolio
from src.covariance       import rolling_covariance
from src.monte_carlo_var  import run_monte_carlo_experiment
from src.latency          import summarize_latency

## 1. Load Prerequisites

In [ ]:
daily    = load_saved_data('daily_prices.csv',         config.DATA_RAW)
intraday = load_saved_data('intraday_1min_prices.csv', config.DATA_RAW)

log_returns = compute_log_returns(daily)
portfolio   = initialize_portfolio(intraday.iloc[0])
shares      = portfolio['shares']
cov_roll    = rolling_covariance(log_returns)

## 2. Monte Carlo VaR Benchmark

In [ ]:
# Run MC VaR at the first bar of the intraday session
# run_monte_carlo_experiment tests all path counts in config.MC_PATHS
mc_results = run_monte_carlo_experiment(
    shares       = shares,
    current_prices = intraday.iloc[0],
    cov_matrix   = cov_roll,
    path_counts  = config.MC_PATHS,
    save         = True,
)
mc_results

## 3. Latency vs Path Count

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

path_counts = mc_results['n_paths'].values

axes[0].bar(path_counts.astype(str), mc_results['latency_ms'], color='steelblue')
axes[0].set_xlabel('Path Count')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('MC VaR Latency vs Path Count')

axes[1].bar(path_counts.astype(str), mc_results['var_99_mc'], color='crimson', label='MC 99% VaR')
axes[1].axhline(mc_results['var_99_parametric'].iloc[0], color='black',
                linestyle='--', label='Parametric 99% VaR')
axes[1].set_xlabel('Path Count')
axes[1].set_ylabel('VaR ($)')
axes[1].set_title('MC VaR Estimate vs Path Count')
axes[1].legend()

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/latency_comparison.png', dpi=150)
plt.show()